# Polymarket Iran Markets: Analysis\n\nWe're going to dig into the 33 Iran-related prediction markets on Polymarket, covering two conflict periods: the **June 2025 12-Day War** and the **February 2026 U.S. Strikes**.\n\nThe goal is to figure out:\n- How much total money flowed through these markets\n- How volume splits between the two conflict periods\n- Which markets attracted the most money\n- How concentrated the betting is\n\nData was collected via the Polymarket Gamma API on March 1, 2026.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nimport matplotlib.ticker as mticker\n\npd.set_option('display.float_format', '{:,.2f}'.format)\npd.set_option('display.max_colwidth', 60)\n\ndf = pd.read_csv('polymarket_iran_markets.csv')\nprint(f"Shape: {df.shape}")\nprint(f"Columns: {list(df.columns)}")\ndf.head(3)

## How much total money are we talking about?\n\nLet's add up all the trading volume across every Iran market on Polymarket. Then we'll break it down by conflict period.

In [ ]:
total_volume = df['Total Volume ($)'].sum()\nprint(f"Total volume across ALL 33 Iran markets: ${total_volume:,.2f}")\nprint()\n\n# Break it down by conflict period\nperiod_vol = df.groupby('Event Period')['Total Volume ($)'].agg(['sum', 'count']).sort_values('sum', ascending=False)\nperiod_vol.columns = ['Total Volume', 'Markets']\nperiod_vol['Total Volume (formatted)'] = period_vol['Total Volume'].apply(lambda x: f"${x:,.0f}")\nprint(period_vol[['Markets', 'Total Volume (formatted)']])\nprint()\n\n# The two main periods\nfeb_2026 = df[df['Event Period'] == '2026 US Strikes']['Total Volume ($)'].sum()\njune_2025 = df[df['Event Period'] == 'June 2025 12-Day War']['Total Volume ($)'].sum()\nprint(f"Feb 2026 U.S. Strikes: ${feb_2026:,.0f} ({df[df['Event Period']=='2026 US Strikes'].shape[0]} markets)")\nprint(f"June 2025 12-Day War:  ${june_2025:,.0f} ({df[df['Event Period']=='June 2025 12-Day War'].shape[0]} markets)")\nprint(f"\\nIncrease factor: {feb_2026/june_2025:.1f}x")

So we get **$943 million** for the 2026 strikes period and **$175 million** for the June 2025 war. That's a **5.4x increase**. Bettors took the 2026 escalation far more seriously.\n\n## Top 10 markets by volume\n\nWhich individual markets attracted the most money? And are they single yes/no bets or multi-outcome aggregates with dozens of sub-markets?

In [ ]:
top10 = df.nlargest(10, 'Total Volume ($)')[['Market Title', 'Event Period', 'Total Volume ($)', 'Market Type', 'Sub-Markets', 'Status']].copy()\ntop10['Volume'] = top10['Total Volume ($)'].apply(lambda x: f"${x/1e6:,.1f}M")\ntop10.index = range(1, 11)\ntop10[['Market Title', 'Event Period', 'Volume', 'Market Type', 'Sub-Markets']]

The biggest single market, "US strikes Iran by...?", is actually an aggregate of **65 sub-markets** betting on different dates. It's important to distinguish these from single binary yes/no bets.\n\n## Volume concentration: how top-heavy is the betting?\n\nLet's see what percentage of total volume is held by the top 1, 3, 5, and 10 markets.

In [ ]:
sorted_vol = df.sort_values('Total Volume ($)', ascending=False)['Total Volume ($)'].values\ntotal = sorted_vol.sum()\n\nfor n in [1, 3, 5, 10]:\n    top_n = sorted_vol[:n].sum()\n    print(f"Top {n:>2} markets: ${top_n/1e6:>8,.1f}M = {top_n/total*100:.1f}% of total")\n\nremaining = total - sorted_vol[:10].sum()\nprint(f"\\nRemaining {len(sorted_vol)-10} markets: ${remaining/1e6:,.1f}M = {remaining/total*100:.1f}% of total")

The betting is extremely top-heavy: the **top 10 markets hold ~82% of all volume**. The remaining 23 markets share just ~18%.\n\n## Volume by category: what are people actually betting on?\n\nLet's categorize each market by topic and compare how the June 2025 and Feb 2026 periods differ in *what* people bet on.

In [ ]:
def categorize(title):\n    t = title.lower()\n    if any(w in t for w in ['strike', 'military action', 'invade', 'war']):\n        return 'Military strikes'\n    if any(w in t for w in ['khamenei', 'supreme leader', 'leader of iran']):\n        return 'Leadership'\n    if any(w in t for w in ['ceasefire', 'conflict ends', 'diplomatic', 'meeting']):\n        return 'Ceasefire / Peace'\n    if any(w in t for w in ['nuclear', 'fordow']):\n        return 'Nuclear / Facilities'\n    if any(w in t for w in ['regime fall', 'regime change', 'democracy']):\n        return 'Regime change'\n    if any(w in t for w in ['pahlavi', 'recogniz']):\n        return 'Pahlavi / Succession'\n    return 'Other'\n\ndf['Category'] = df['Market Title'].apply(categorize)\n\n# Only look at the two main conflict periods\nmain = df[df['Event Period'].isin(['June 2025 12-Day War', '2026 US Strikes'])]\ncat_period = main.groupby(['Category', 'Event Period'])['Total Volume ($)'].sum().unstack(fill_value=0)\ncat_period.columns = ['2026 US Strikes', 'June 2025']\ncat_period['Total'] = cat_period.sum(axis=1)\ncat_period = cat_period.sort_values('Total', ascending=False)\n\n# Format for display\nfor col in cat_period.columns:\n    cat_period[f'{col} (fmt)'] = cat_period[col].apply(lambda x: f"${x/1e6:,.1f}M" if x > 0 else "—")\n\ncat_period[['June 2025 (fmt)', '2026 US Strikes (fmt)', 'Total (fmt)']]

The shift is dramatic. In June 2025, **ceasefire bets dominated** (people wanted to know when it would end). In 2026, **military strike timing** dominates, and **leadership bets exploded**.\n\n## Chart: Volume by conflict period\n\nLet's visualize the two periods side by side.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))\n\nperiods = ['June 2025\\n12-Day War', 'Feb 2026\\nU.S. Strikes']\nvolumes = [june_2025 / 1e6, feb_2026 / 1e6]\ncolors = ['#dc2626', '#2563eb']\n\nbars = ax.barh(periods, volumes, color=colors, height=0.5)\nfor bar, v in zip(bars, volumes):\n    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,\n            f'${v:,.0f}M', va='center', fontsize=11, fontweight='bold')\n\nax.set_xlim(0, max(volumes) * 1.15)\nax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))\nax.set_title('Total trading volume by conflict period', fontsize=13, fontweight='bold', loc='left')\nax.spines[['top', 'right']].set_visible(False)\nax.tick_params(left=False)\nplt.tight_layout()\nplt.show()

## Chart: Top 10 markets\n\nLet's visualize the biggest markets. We'll color single binary markets differently from multi-outcome aggregates to be transparent about what we're comparing.

In [ ]:
top10_data = df.nlargest(10, 'Total Volume ($)').iloc[::-1]  # reverse for horizontal bar\n\nfig, ax = plt.subplots(figsize=(9, 5))\ncolors = ['#2563eb' if t == 'Single binary' else '#93c5fd' for t in top10_data['Market Type']]\nbars = ax.barh(range(len(top10_data)), top10_data['Total Volume ($)'] / 1e6, color=colors, height=0.6)\n\n# Labels\nfor i, (_, row) in enumerate(top10_data.iterrows()):\n    label = row['Market Title'][:35] + '...' if len(row['Market Title']) > 35 else row['Market Title']\n    ax.text(-5, i, label, ha='right', va='center', fontsize=9)\n    ax.text(row['Total Volume ($)']/1e6 + 5, i, f\"${row['Total Volume ($)']/1e6:,.0f}M\", va='center', fontsize=9, fontweight='bold')\n\nax.set_yticks([])\nax.set_xlim(-220, 600)\nax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M' if x >= 0 else ''))\nax.set_title('Top 10 Polymarket Iran markets by volume', fontsize=13, fontweight='bold', loc='left')\nax.spines[['top', 'right', 'left']].set_visible(False)\n\n# Legend\nfrom matplotlib.patches import Patch\nax.legend(handles=[Patch(color='#2563eb', label='Single binary'), Patch(color='#93c5fd', label='Multi-outcome aggregate')],\n          loc='lower right', fontsize=9, frameon=False)\nplt.tight_layout()\nplt.show()

## Chart: Volume by category, June 2025 vs. Feb 2026\n\nThis is the grouped bar chart that shows how the *type* of betting shifted between the two conflicts.

In [ ]:
import numpy as np\n\ncats = cat_period.index.tolist()\njune_vals = cat_period['June 2025'].values / 1e6\nfeb_vals = cat_period['2026 US Strikes'].values / 1e6\n\ny = np.arange(len(cats))\nh = 0.35\n\nfig, ax = plt.subplots(figsize=(9, 5))\nax.barh(y + h/2, june_vals, h, color='#dc2626', alpha=0.85, label='June 2025')\nax.barh(y - h/2, feb_vals, h, color='#2563eb', alpha=0.85, label='Feb 2026')\n\nax.set_yticks(y)\nax.set_yticklabels(cats, fontsize=10)\nax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))\nax.set_title('Volume by market category: how the bets changed', fontsize=13, fontweight='bold', loc='left')\nax.spines[['top', 'right']].set_visible(False)\nax.legend(fontsize=10, frameon=False)\nax.invert_yaxis()\nplt.tight_layout()\nplt.show()

## Active vs. Closed markets

In [ ]:
status = df.groupby('Status').agg(\n    Markets=('Market Title', 'count'),\n    Volume=('Total Volume ($)', 'sum')\n)\nstatus['Volume (fmt)'] = status['Volume'].apply(lambda x: f"${x/1e6:,.1f}M")\nprint(status[['Markets', 'Volume (fmt)']])\nprint(f"\\nActive markets with 24h volume > $0: {(df['24h Volume ($)'] > 0).sum()}")

## Key findings\n\nHere's what feeds into the scrollytelling visualization:\n\n1. **$943 million** in total Polymarket volume for the 2026 U.S. Strikes period (22 markets)\n2. **$175 million** for the June 2025 12-Day War (9 markets)\n3. That's a **5.4x increase** between the two conflict periods\n4. The **top 10 markets hold ~82%** of all trading volume\n5. **Military strike timing** dominates 2026 betting (vs. ceasefire bets in 2025)\n6. **Leadership/Khamenei markets exploded** from $8M to $221M between periods\n7. The single largest market ("US strikes Iran by...?") is a 65-sub-market aggregate worth $529M\n8. Most June 2025 markets are now **closed** and resolved; many 2026 markets remain **active**